# Neptune Graph Visualization
## Biomedical Knowledge Graph — Drug-Disease Relationships

This notebook uses the AWS `graph-notebook` to visualize our Neptune graph data.
- **Neptune DB**: Gremlin queries with `%%gremlin` magic
- **Neptune Analytics**: openCypher queries with `%%oc` magic
- Interactive **Graph tab** renders force-directed layouts

---
## 1. Setup & Connection

In [ ]:
# Load graph-notebook magic commands
%load_ext graph_notebook.magics

In [ ]:
# View current connection config
%graph_notebook_config

In [ ]:
# Check Neptune DB connection status
%status

---
## 2. Neptune DB — Gremlin Graph Visualization

The `%%gremlin` magic sends Gremlin queries to Neptune DB.
Any query returning `.path()` gets an interactive **Graph tab** with force-directed rendering.

In [ ]:
%%gremlin

// Count all vertices and edges
g.V().groupCount().by(label)

In [ ]:
%%gremlin

// Count relationships
g.E().groupCount().by(label)

### 2a. Visualize ALL Drug → Disease Relationships
Uses `.path()` to enable the Graph tab visualization.
The `--path-pattern` hint tells the visualizer what the path structure looks like.

In [ ]:
%%gremlin -p v,oute,inv

g.V().hasLabel('Drug')
 .outE('TREATS')
 .inV()
 .path()
 .by('name')
 .by(label)
 .by('name')

### 2b. Visualize with Grouping (Color by Label)
The `--group-by` flag colors vertices by a property — here we group by vertex label.

In [ ]:
%%gremlin -p v,oute,inv -g T.label

g.V().hasLabel('Drug')
 .outE('TREATS')
 .inV()
 .path()
 .by(elementMap())
 .by(elementMap())
 .by(elementMap())

### 2c. Single Drug Deep-Dive: Pembrolizumab
Show which diseases Pembrolizumab treats — PD-1 immune checkpoint inhibitor.

In [ ]:
%%gremlin -p v,oute,inv

g.V().has('Drug', 'name', 'Pembrolizumab')
 .outE('TREATS')
 .inV()
 .path()
 .by(elementMap())
 .by(elementMap())
 .by(elementMap())

### 2d. Disease-Centric View: Which Drugs Treat Lung Cancer?

In [ ]:
%%gremlin -p v,ine,outv

g.V().has('Disease', 'name', 'Non-Small Cell Lung Cancer')
 .inE('TREATS')
 .outV()
 .path()
 .by(elementMap())
 .by(elementMap())
 .by(elementMap())

### 2e. Drug Properties Table

In [ ]:
%%gremlin

g.V().hasLabel('Drug')
 .project('name', 'mechanism', 'type')
 .by('name')
 .by('mechanism')
 .by('drug_type')

### 2f. Shared-Disease Drugs (Co-Indication Network)
Find drugs that treat the same disease — critical for combination therapy analysis.

In [ ]:
%%gremlin -p v,oute,inv,ine,outv

g.V().hasLabel('Drug').as('drug1')
 .outE('TREATS').inV().as('disease')
 .inE('TREATS').outV().as('drug2')
 .where('drug1', neq('drug2'))
 .path()
 .by(elementMap())
 .by(elementMap())
 .by(elementMap())
 .by(elementMap())
 .by(elementMap())
 .dedup()

---
## 3. Neptune Analytics — openCypher Visualization

Switch connection to Neptune Analytics graph for native vector + graph queries.

**Note:** To connect to Neptune Analytics, update the config below:

In [ ]:
%%graph_notebook_config
{
  "host": "g-0l15plmr0b.us-west-2.neptune-graph.amazonaws.com",
  "neptune_service": "neptune-graph",
  "port": 443,
  "auth_mode": "IAM",
  "ssl": true,
  "aws_region": "us-west-2"
}

In [ ]:
%%oc

MATCH (d:Drug)-[r:TREATS]->(dis:Disease)
RETURN d, r, dis

In [ ]:
%%oc

MATCH (d:Drug)
RETURN d.name AS drug, d.mechanism AS mechanism, d.drug_type AS type, d.approval_status AS status
ORDER BY d.name

### 3a. Vector Similarity Search + Graph Traversal (Unified)
The "holy grail" — native vector search and graph traversal in a single query.

In [ ]:
%%oc

// Find drugs similar to a PD-1 inhibitor query vector, then traverse to diseases
CALL neptune.algo.vectors.topKByNode(n, {topK: 5})
  WITH n WHERE n:Drug AND n.name = 'Pembrolizumab'
YIELD node, score
MATCH (node)-[:TREATS]->(disease:Disease)
RETURN node.name AS drug, score, collect(DISTINCT disease.name) AS diseases
ORDER BY score DESC

### 3b. Find Similar Drugs by Vector Embedding

In [ ]:
%%oc

// Find top 5 drugs most similar to Pembrolizumab by embedding
MATCH (anchor:Drug {name: 'Pembrolizumab'})
CALL neptune.algo.vectors.topKByNode(anchor, {topK: 5})
YIELD node, score
RETURN node.name AS similar_drug, node.mechanism AS mechanism, score
ORDER BY score DESC

---
## 4. Switch Back to Neptune DB
Restore the original Neptune DB connection.

In [ ]:
%%graph_notebook_config
{
  "host": "graphrag-neptune-benchmark.cluster-c52uokq6k2wj.us-west-2.neptune.amazonaws.com",
  "neptune_service": "neptune-db",
  "port": 8182,
  "auth_mode": "IAM",
  "ssl": true,
  "aws_region": "us-west-2"
}

---
## 5. Graph Statistics Summary

In [ ]:
%%gremlin

g.V().group().by(label).by(count())

In [ ]:
%%gremlin

// Drug degree distribution — how many diseases each drug treats
g.V().hasLabel('Drug')
 .project('drug', 'disease_count')
 .by('name')
 .by(out('TREATS').count())
 .order().by('disease_count', desc)